In [1]:
from pydantic import BaseModel
from typing import Literal
import time

class TradeRecord(BaseModel):
    symbol:    str
    timestamp: int
    price:     float
    qty:    float
    side:      Literal['buy', 'sell']

class OrderBookSnapshot(BaseModel):
    symbol:    str
    timestamp: int
    bids:      list[list[float]]
    asks:      list[list[float]]


In [2]:
import asyncio, json, websockets
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode()
)

async def stream_order_book(symbol: str = 'btcusdt'):
    # @depth20@100ms = 20-level book, pushed every 100ms
    url = f"wss://stream.binance.com:9443/ws/{symbol}@depth20@100ms"

    async with websockets.connect(url, ping_interval=20) as ws:
        async for raw in ws:
            msg = json.loads(raw)
            record = {
                'symbol':    symbol.upper(),
                'timestamp': int(time.time() * 1000),           # event time ms
                'bids':      msg['bids'],                       # [[price, qty], ...]
                'asks':      msg['asks'],
                'source':    'binance_ws'
            }
            record = OrderBookSnapshot(**record)
            producer.send('raw_order_book', value=record.model_dump())

async def stream_trades(symbol: str = 'btcusdt'):
    url = f"wss://stream.binance.com:9443/ws/{symbol}@trade"

    async with websockets.connect(url, ping_interval=20) as ws:
        async for raw in ws:
            msg = json.loads(raw)
            record = {
                'symbol':    symbol.upper(),
                'timestamp': msg['T'],
                'price':     float(msg['p']),
                'qty':       float(msg['q']),
                'side':      'buy' if not msg['m'] else 'sell' ,
            }
            record = TradeRecord(**record)
            producer.send('raw_prices', value=record.model_dump())

async def main():
    await asyncio.gather(
        stream_order_book('btcusdt'),
        stream_trades('btcusdt'),
    )


In [ ]:
from kafka import KafkaConsumer

def value_deserializer(value):
    return json.loads(value.decode())

consumer = KafkaConsumer(
    'raw_order_book',
    bootstrap_servers='localhost:9092',
    auto_offset_reset = 'latest',
    group_id = 'test_order_book_consumer',
    value_deserializer=value_deserializer
)

for msg in consumer:
    print(msg.value)

{'symbol': 'BTCUSDT', 'timestamp': 1775543779964, 'bids': [[68490.37, 0.38798], [68490.36, 0.00064], [68490.04, 8e-05], [68490.0, 0.00048], [68489.99, 0.00638], [68489.7, 8e-05], [68489.39, 8e-05], [68488.52, 0.00028], [68488.47, 0.00117], [68488.32, 8e-05], [68488.18, 0.00011], [68488.0, 0.0008], [68487.53, 9e-05], [68487.11, 8e-05], [68487.1, 8e-05], [68486.98, 8e-05], [68486.97, 0.01175], [68486.81, 0.003], [68486.7, 0.00394], [68486.47, 0.00024]], 'asks': [[68490.38, 5.07192], [68490.39, 8e-05], [68490.4, 0.003], [68490.55, 0.01503], [68491.06, 8e-05], [68492.11, 0.00738], [68492.12, 0.003], [68492.23, 0.00797], [68492.44, 8e-05], [68493.38, 8e-05], [68493.67, 2.43157], [68493.68, 0.22707], [68493.84, 0.003], [68493.96, 0.05008], [68493.97, 7.95009], [68494.41, 0.02], [68494.61, 0.0073], [68494.62, 0.16786], [68494.94, 0.0146], [68495.56, 0.003]]}
{'symbol': 'BTCUSDT', 'timestamp': 1775543780058, 'bids': [[68490.37, 0.38798], [68490.36, 0.00064], [68490.04, 8e-05], [68490.0, 0.0004